In [ ]:
using Revise

using Pkg
Pkg.activate("..")

In [ ]:
using bslLD,Plots, Statistics, ProgressBars, FFTW
using Polynomials, LinearAlgebra, SpecialPolynomials,Plots, CubicSplines
using DSP

In [ ]:
function initialize(grid, finit)
    f = bslLD.Distribution(grid, 0)
    for (ix, x) in enumerate(grid.xaxes[1])
        for (iv1, v1) in enumerate(grid.vaxes[1])
            for (iv2, v2) in enumerate(grid.vaxes[2])
                if typeof(grid) <: bslLD.PolarGrid
                        vx = v1 * cos(v2)
                        vy = v1 * sin(v2)
                        f.data[ix, iv1, iv2] = finit(x, vx, vy)
                else
                        f.data[ix, iv1, iv2] = finit(x, v1,v2)
                end
            end
        end
    end
    return f
end

In [ ]:
grid =  bslLD.Grid([0,-4.0,-4.0],[10*pi,4.0,4.0],[32,8,8],0.5,100,1; type=bslLD.Polar)
finit1(x,vx,vy) = (1-cos((x-vy)/5)) * exp(-0.5*((vx)^2 + vy^2))
f = initialize(grid, finit1)
e = bslLD.empty_vectorfield(grid)
@time bslLD.advectV!(f,grid,e)

In [ ]:
psinit  = []
psfinal = []
ptrack = []
ppos = []

for type in (bslLD.Polar,bslLD.Cart)

    track = []

    grid =  bslLD.Grid([0,-4.0,-4.0],[10*pi,4.0,4.0],[32,16,16],0.1,500,1; type=type)
    finit1(x,vx,vy) = (1-0.5cos((x)/5)) * exp(-0.5*((vx-1)^2 + vy^2))
    f = initialize(grid, finit1)
    p = bslLD.heatmap_fv(f.data[1,:,:], grid)
    push!(psinit, p)

    e = bslLD.empty_vectorfield(grid)
    e.data[2] .= 0.1

    @time bslLD.advectV!(f, grid, e)


    for grid.index[1]= grid.itime
        bslLD.advectX!(f,grid)
        bslLD.advectV!(f, grid, e)
        if mod(grid.index[1],10) != 0
            continue
        end   
        rho = bslLD.compute_density(f,grid)
        push!(track, rho)    
    end

    p = bslLD.heatmap_fv(f.data[1,:,:], grid)
    push!(psfinal, p)
    push!(ptrack, plot(track, legend = false))
    push!(ppos, plot(map(x->grid.xaxes[1][argmax(x)]-grid.xaxes[1][argmax(track[1])], track) ))
end


plot([psinit; psfinal; ptrack; ppos]..., layout = (4,2), size = (600, 800))


In [ ]:
psinit  = []
psfinal = []
ptrack = []
ppos = []

Nx = 32
Nt = 10000

for type in (bslLD.Polar,bslLD.Cart)

    track = []

    grid =  bslLD.Grid([0,-4.0,-4.0],[10*pi,4.0,4.0],[Nx,8,8],0.1,Nt,1; type=type)
    finit1(x,vx,vy) = (1-0.000001*rand()) * exp(-0.5*((vx)^2 + vy^2))
    f = initialize(grid, finit1)
    p = bslLD.heatmap_fv(f.data[1,:,:], grid)
    push!(psinit, p)

    e = bslLD.empty_vectorfield(grid)
    e.data[2] .= 0.1

    @time bslLD.advectV!(f, grid, e)


    for grid.index[1]= grid.itime
        bslLD.advectX!(f,grid)
        rho = bslLD.compute_density(f,grid)
        phi = bslLD.adiabatic(bslLD.ScalarField(rho))
        e = bslLD.compute_e(phi, grid)
        bslLD.advectV!(f, grid, e)
        push!(track, rho .- mean(rho))    
    end
    p = bslLD.heatmap_fv(f.data[1,:,:], grid)
    push!(psfinal, p)

    track_m  = reduce(hcat, track) 
    track_m = (track_m' .* Windows.kaiser(size(track_m)[2],14))'
    push!(ptrack, heatmap(log.(abs.(fft(track_m)))[1:16,1:500]', legend = false))
end


plot([psinit; psfinal;ptrack]..., layout = (4,2), size = (600, 800))


In [ ]:
N = 7

function hermite_projection(x_data, f_data, N)
    H = Polynomials.basis.(Hermite, 0:N-1)

    weights = @. exp(-x_data^2 / 2)

    B = hcat([p.(x_data) for p in H]...)

    W_sqrt = Diagonal(sqrt.(weights))
    B_w = W_sqrt * B
    f_w = W_sqrt * f_data

    c = B_w \ f_w

    sum(c .* H)
end

In [ ]:
using Plots


grid =  bslLD.Grid([0,-4.0,-4.0],[10*pi,4.0,4.0],[Nx,132,32],0.1,Nt,1; type=bslLD.Polar)
finit1(x,vx,vy) = exp(-0.5*((vx)^2 + vy^2)) * sin(atan(vy,vx))
f = initialize(grid, finit1)


imax = 5
F = bslLD.fft(f.data[:, :, :], [3])/(2*pi)
p = plot(layout = (imax,1), size = (600, 1500),left_margin = 12Plots.mm)


for im in 1:imax
    splineRe = CubicSpline(grid.vaxes[1], real.(F[1,:, im]))
    splineIm = CubicSpline(grid.vaxes[1], imag.(F[1,:, im]))

    polyRe = hermite_projection(grid.vaxes[1], real.(F[1,:, im]),10)
    polyIm = hermite_projection(grid.vaxes[1], imag.(F[1,:, im]),10)

    x_plot = minimum(grid.vaxes[1]):0.1:maximum(grid.vaxes[1])

    plot!(p[im],grid.vaxes[1], real.(F[1,:, im]), label = "real")
    plot!(p[im], polyRe, 0,maximum(grid.vaxes[1]),label = "hermite fit real", lw=2, ls=:dash)
    plot!(p[im],grid.vaxes[1], imag.(F[1,:, im]), label = "imag")
    plot!(p[im], polyIm, 0,maximum(grid.vaxes[1]),label = "hermite fit imag", lw=2, ls=:dash)
    plot!(p[im], x_plot, splineRe[x_plot],label = "spline inter imag", lw=2, ls=:dot)
    plot!(p[im], x_plot, splineIm[x_plot],label = "pline inter imag", lw=2, ls=:dot)
    plot!(p[im], title = "im = $(im-1)")
end
p

In [ ]:
 bslLD.heatmap_fv(f.data[1,:,:], grid)

In [ ]:
heatmap(grid.vaxes[2], grid.vaxes[1],f0)

In [ ]:
reconstruct_f_dierckx_displaced!(grid,f,5)
heatmap(grid.vaxes[2], grid.vaxes[1], f.data[2,:,:])

In [ ]:
p1 = plot(
    log.(abs.(bslLD.fft(f0, [2]))),
    title = "f0 spectrum",
    legend = false
)

p2 = plot(
    log.(abs.(bslLD.fft(f.data[2, :, :], [2])) ),
    title = "f data spectrum",
    legend = false,
    ls = :dash
)


p3 = plot(
    (abs.(bslLD.fft(f.data[2, :, :], [2])) ./ abs.(bslLD.fft(f0, [2]))),
    title = "f data spectrum",
    legend = false,
    ls = :dash
)


plot(p1, p2, p3, layout = (1, 3))



In [ ]:
grid.vaxes

In [ ]:
grid = bslLD.Grid([0,-1,-1.0],[10.0,1.0,1.0],[16,4,32],0.5,100,1; type=bslLD.Polar)

rp = zeros(length(grid.vaxes[1]) , length(grid.vaxes[2]))
phip = zeros(length(grid.vaxes[1]) , length(grid.vaxes[2]))

ri = zeros(length(grid.vaxes[1]) , length(grid.vaxes[2]))
phii = zeros(length(grid.vaxes[1]) , length(grid.vaxes[2]))

vx = zeros(length(grid.vaxes[1]) , length(grid.vaxes[2]))
vy = zeros(length(grid.vaxes[1]) , length(grid.vaxes[2]))

vxi = zeros(length(grid.vaxes[1]) , length(grid.vaxes[2]))
vyi = zeros(length(grid.vaxes[1]) , length(grid.vaxes[2]))

vrp_ref = Ref(0.0)
phip_ref = Ref(0.0)

for (ir, r) in enumerate(grid.vaxes[1])
    for (iphi, phi) in enumerate(grid.vaxes[2])
        dx, dy = (-1,0)
        bslLD.displace_polar_velocity!(vrp_ref, phip_ref, r, phi,dx,dy)
        rp[ir, iphi], phip[ir, iphi] = (vrp_ref[], phip_ref[])
        ri[ir, iphi], phii[ir, iphi] = (grid.vaxes[1][ir], grid.vaxes[2][iphi])


        vx[ir, iphi], vy[ir, iphi] = (rp[ir, iphi]*cos(phip[ir, iphi]),
                                    rp[ir, iphi]*sin(phip[ir, iphi]))

        vxi[ir, iphi], vyi[ir, iphi] = (ri[ir, iphi]*cos(phii[ir, iphi]),
                                      ri[ir, iphi]*sin(phii[ir, iphi]))
    end
end

p1 = plot(ri', phii', legend = false, aspect_ratio = :equal, xlabel = "r", ylabel = "phi")
p1 = plot!(rp', phip' .+ pi, apha = 0.5, legend = false, aspect_ratio = :equal)

# p1 = quiver!(ri, phii, quiver = (rp-ri,phip-phii), legend = false, aspect_ratio = :equal)

p2 = plot(vxi', vyi', legend = false, aspect_ratio = :equal, xlabel = "vx", ylabel = "vy")
p2 = plot!(vx', vy', apha = 0.5, legend = false, aspect_ratio = :equal)

# p2 = quiver!(vxi, vyi, quiver = (vx-vxi,vy-vyi), legend = false, aspect_ratio = :equal)


plot(p1, p2, layout = (1, 2), size = (800,400))




In [ ]:

p1 = scatter(ri, phii, legend = false, aspect_ratio = :equal, xlabel = "r", ylabel = "phi")
p1 = scatter!(rp, phip .+ pi, apha = 0.5, legend = false, aspect_ratio = :equal)

# p1 = quiver!(ri, phii, quiver = (rp-ri,phip-phii), legend = false, aspect_ratio = :equal)

p2 = scatter(vxi, vyi, legend = false, aspect_ratio = :equal, xlabel = "vx", ylabel = "vy")
p2 = scatter!(vx, vy, apha = 0.5, legend = false, aspect_ratio = :equal)

# p2 = quiver!(vxi, vyi, quiver = (vx-vxi,vy-vyi), legend = false, aspect_ratio = :equal)


plot(p1, p2, layout = (1, 2), size = (800,400))


In [ ]:
using Dierckx
using Base.Threads
using LinearAlgebra

function get_displacement(r, vr, phi)
    # This now defines a bulk velocity shift (dvx, dvy)
    # based on the phase-space location (r, vr, phi).
    # Assuming constant bulk shift for simplicity:
    delta_vx = 0.1
    delta_vy = 0.1
    return delta_vx, delta_vy
end

function displace_polar_velocity(vr::Real, phi::Real, delta_vx::Real, delta_vy::Real)
    # Convert polar velocity (vr, phi) to Cartesian velocity (vx, vy)
    vx = vr * cos(phi)
    vy = vr * sin(phi)
    
    # Apply Cartesian velocity displacement (bulk flow shift)
    vx_prime = vx + delta_vx
    vy_prime = vy + delta_vy
    
    # Convert back to polar velocity (vr', phi')
    vr_prime = sqrt(vx_prime^2 + vy_prime^2)
    phi_prime = atan(vy_prime, vx_prime)
    return (vr_prime, phi_prime)
end


function reconstruct_f_dierckx_displaced!(grid, dist, imax)
    F = bslLD.fft(dist.data[:, :, :], [3]) / length(grid.vaxes[2])

    rgrid   = grid.xaxes[1] # Spatial radius 'r'
    vgrid   = grid.vaxes[1] # Radial velocity 'vr'
    phigrid = grid.vaxes[2] # Angular velocity coordinate 'phi'

    nx   = length(rgrid)
    nv   = length(vgrid)
    nphi = length(phigrid)

    imax = min(imax, fld(nphi, 2))

    @threads for ix in 1:nx
        r_current = rgrid[ix]

        itp_re = Vector{Dierckx.Spline1D}(undef, imax+1)
        itp_im = Vector{Dierckx.Spline1D}(undef, imax+1)

        @inbounds for m in 1:imax+1
            vals_re = real.(F[ix, :, m])
            vals_im = imag.(F[ix, :, m])
            # Splines interpolate Fourier coefficients F_m(v_r) 
            itp_re[m] = Spline1D(vgrid, vals_re, k=3)
            itp_im[m] = Spline1D(vgrid, vals_im, k=3)
        end

        # Pre-calculate displaced polar velocity coordinates for the slice
        vr_prime_slice = Array{Float64}(undef, nv, nphi)
        phi_prime_slice = Array{Float64}(undef, nv, nphi)

        @inbounds for iv in 1:nv
            vr_grid = vgrid[iv]
            @inbounds for ip in 1:nphi
                phi_grid = phigrid[ip]

                delta_vx, delta_vy = get_displacement(r_current, vr_grid, phi_grid)
                
                # Transformation applies to the velocity coordinates
                vr_prime, phi_prime = displace_polar_velocity(vr_grid, phi_grid, delta_vx, delta_vy)

                vr_prime_slice[iv, ip] = vr_prime
                phi_prime_slice[iv, ip] = phi_prime
            end
        end

        # Reconstruct by evaluating F_m at v_r' and using phi'
        @inbounds for iv in 1:nv
            # The velocity value to sample is the *displaced* radial velocity v_r'
            # This requires sampling the spline at multiple v_r' points, 
            # one for each ip, so we must calculate Fvals inside the ip loop.

            @inbounds for ip in 1:nphi
                vr_prime = vr_prime_slice[iv, ip]
                phi_prime = phi_prime_slice[iv, ip]

                # Evaluate all splines at the displaced radial velocity vr_prime
                FvalsRe_at_vr_prime = Array{Float64}(undef, imax+1)
                FvalsIm_at_vr_prime = Array{Float64}(undef, imax+1)
                
                @simd for m in 1:imax+1
                    FvalsRe_at_vr_prime[m] = itp_re[m](vr_prime)
                    FvalsIm_at_vr_prime[m] = itp_im[m](vr_prime)
                end

                # Perform reconstruction sum
                r_val = FvalsRe_at_vr_prime[1] 

                @inbounds for m in 1:imax
                    re = FvalsRe_at_vr_prime[m+1]
                    im = FvalsIm_at_vr_prime[m+1]
                    
                    m_phi_prime = m * phi_prime
                
                    r_val += 2.0 * (re * cos(m_phi_prime) - im * sin(m_phi_prime))
                end

                dist.data[ix, iv, ip] = r_val
            end
        end
    end

    return dist
end

In [ ]:
f, grid = initialize()

p1 = bslLD.heatmap_fv(f.data[1,:,:], grid)
e = bslLD.empty_vectorfield(grid)
e.data[1] .= 1.0

bslLD.advectV!(f, grid, e)

p2 = bslLD.heatmap_fv(f.data[1,:,:], grid)

plot(p1, p2, layout = (1,2))

In [ ]:
@time bslLD.advectV!(f, grid, e)
@time reconstruct_f_dierckx_displaced!(grid,f,5)

In [ ]:
finit1(x,vx,vy) = (1+0.1sin(-2. /5. * (x-vy))) * exp(-0.5*((vx)^2 + vy^2))
f, grid = initialize(finit1)

p1 = bslLD.heatmap_fv(f.data[1,:,:], grid)


In [ ]:
rho = bslLD.compute_density(f,grid)
p = plot(grid.xaxes[1],rho,legend = false)
track = [rho]

e = bslLD.empty_vectorfield(grid)

for grid.index[1]= grid.itime
    bslLD.advectX!(f,grid)
    rho = bslLD.compute_density(f,grid)
    bslLD.advectV!(f, grid, e)
    p = plot!(grid.xaxes[1],rho)
    push!(track, rho)
end
display(p)

In [ ]:
plot(grid.time,map(x->grid.xaxes[1][argmax(x)], track) .- grid.xaxes[1][argmax(track[1])])

In [ ]:
p1 = bslLD.heatmap_fv(f.data[1,:,:], grid)
